# 04 - Visualization Dashboard
## MA Waterways Heatwave Risk Analysis

**Objective**: Create comprehensive visualizations for stakeholder communication

**Visualizations**:
1. Monthly DO seasonal pattern (U-curve)
2. Temperature-DO relationship
3. Summer temperature trends
4. Correlation heatmap
5. Risk score distributions
6. Geographic patterns (if coordinates available)

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.append('../src')

# Import visualization functions
from visualization import (
    plot_monthly_do_pattern,
    plot_temp_do_relationship,
    plot_summer_temp_trend,
    plot_correlation_heatmap,
    plot_risk_distribution,
    create_summary_dashboard
)

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries and modules imported successfully")

## 1. Load Featured Data

In [ ]:
# Load data with features
df = pd.read_csv('../data/processed/water_quality_with_features.csv')

# Convert date columns
date_cols = [col for col in df.columns if 'date' in col.lower()]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print(f"Loaded {len(df):,} records with {len(df.columns)} features")

# Identify key columns
do_col = [c for c in df.columns if 'do' in c.lower() and 'do_' not in c.lower()][0]
temp_col = [c for c in df.columns if 'temp' in c.lower() and 'temp_' not in c.lower()][0]

print(f"\nUsing columns:")
print(f"  DO: {do_col}")
print(f"  Temperature: {temp_col}")

## 2. Monthly DO Pattern

Seasonal U-curve showing lowest DO in summer months.

In [ ]:
# Create monthly DO pattern plot
fig, ax = plot_monthly_do_pattern(df, 
                                   do_column=do_col,
                                   save_path='../outputs/figures/monthly_do_pattern.png')
plt.show()

## 3. Temperature-DO Relationship

Scatterplot showing negative correlation between temperature and dissolved oxygen.

In [ ]:
# Create temperature-DO relationship plot
fig, ax = plot_temp_do_relationship(df,
                                    temp_column=temp_col,
                                    do_column=do_col,
                                    save_path='../outputs/figures/temp_do_relationship.png')
plt.show()

## 4. Summer Temperature Trends

Long-term warming trend in summer water temperatures.

In [ ]:
# Create summer temperature trend plot
fig, ax = plot_summer_temp_trend(df,
                                 temp_column=temp_col,
                                 save_path='../outputs/figures/summer_temp_trend.png')
plt.show()

## 5. Correlation Heatmap

Relationships between water quality parameters.

In [ ]:
# Select key numeric columns for correlation
numeric_cols = df.select_dtypes(include=[np.number]).columns
exclude = [col for col in numeric_cols if any(x in col.lower() for x in ['year', 'month', 'day', 'id'])]
corr_cols = [col for col in numeric_cols if col not in exclude][:12]

# Create correlation heatmap
fig, ax = plot_correlation_heatmap(df,
                                   columns=corr_cols,
                                   save_path='../outputs/figures/correlation_heatmap.png')
plt.show()

## 6. Risk Score Distribution

Distribution of composite heat-stress risk scores.

In [ ]:
# Create risk distribution plots
if 'risk_score' in df.columns:
    fig, axes = plot_risk_distribution(df,
                                       risk_column='risk_score',
                                       save_path='../outputs/figures/risk_distribution.png')
    plt.show()
else:
    print("Risk score not available - run feature engineering notebook first")

## 7. Seasonal Comparison

Compare DO and temperature across seasons.

In [ ]:
if 'season' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # DO by season (boxplot)
    season_order = ['Winter', 'Spring', 'Summer', 'Fall']
    sns.boxplot(data=df, x='season', y=do_col, order=season_order, ax=axes[0],
                palette='coolwarm')
    axes[0].axhline(5, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Critical')
    axes[0].set_xlabel('Season', fontweight='bold', fontsize=12)
    axes[0].set_ylabel('Dissolved Oxygen (mg/L)', fontweight='bold', fontsize=12)
    axes[0].set_title('DO by Season', fontweight='bold', fontsize=14)
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Temperature by season
    sns.boxplot(data=df, x='season', y=temp_col, order=season_order, ax=axes[1],
                palette='RdYlBu_r')
    axes[1].axhline(25, color='orange', linestyle='--', linewidth=2, alpha=0.7, label='Warm')
    axes[1].set_xlabel('Season', fontweight='bold', fontsize=12)
    axes[1].set_ylabel('Water Temperature (°C)', fontweight='bold', fontsize=12)
    axes[1].set_title('Temperature by Season', fontweight='bold', fontsize=14)
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../outputs/figures/seasonal_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

## 8. Risk by Year

How has risk changed over time?

In [ ]:
if 'risk_score' in df.columns and 'year' in df.columns:
    yearly_risk = df.groupby('year')['risk_score'].agg(['mean', 'std', 'count']).reset_index()
    
    plt.figure(figsize=(14, 6))
    plt.errorbar(yearly_risk['year'], yearly_risk['mean'], yerr=yearly_risk['std'],
                marker='o', markersize=7, capsize=5, linewidth=2, color='darkred',
                label='Mean Risk Score ± Std Dev')
    
    # Add trend line
    z = np.polyfit(yearly_risk['year'], yearly_risk['mean'], 1)
    p = np.poly1d(z)
    plt.plot(yearly_risk['year'], p(yearly_risk['year']), 'k--', linewidth=2, alpha=0.7,
            label=f'Trend: {z[0]:.3f} pts/year')
    
    plt.xlabel('Year', fontweight='bold', fontsize=12)
    plt.ylabel('Mean Risk Score', fontweight='bold', fontsize=12)
    plt.title('Heat-Stress Risk Score Trend Over Time\n(MA Waterways 2005-2020)',
             fontweight='bold', fontsize=14)
    plt.legend(fontsize=11)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('../outputs/figures/risk_trend_over_time.png', dpi=300, bbox_inches='tight')
    plt.show()

## 9. Combined Stress Analysis

Frequency of compound stress events.

In [ ]:
if 'stress_combo' in df.columns and 'year' in df.columns:
    # Calculate stress events by year
    yearly_stress = df.groupby('year')['stress_combo'].sum().reset_index()
    
    plt.figure(figsize=(14, 6))
    plt.bar(yearly_stress['year'], yearly_stress['stress_combo'],
           color='orangered', edgecolor='black', alpha=0.7)
    
    plt.xlabel('Year', fontweight='bold', fontsize=12)
    plt.ylabel('Number of Combined Stress Events', fontweight='bold', fontsize=12)
    plt.title('Combined Stress Events by Year\n(High Temperature + Low DO)',
             fontweight='bold', fontsize=14)
    plt.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig('../outputs/figures/stress_events_by_year.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\nTotal combined stress events: {df['stress_combo'].sum():,}")
    print(f"Percentage of all records: {df['stress_combo'].mean()*100:.2f}%")

## 10. Summary Dashboard

Generate all key visualizations at once.

In [ ]:
# Create comprehensive dashboard
output_dir = Path('../outputs/figures/dashboard')
output_dir.mkdir(parents=True, exist_ok=True)

create_summary_dashboard(df, output_dir)

print("\n✓ Summary dashboard created successfully!")
print(f"\nFigures saved to: {output_dir}")

## Summary

**Key Visualizations Created**:
1. ✓ Monthly DO seasonal pattern
2. ✓ Temperature-DO relationship
3. ✓ Summer temperature trends
4. ✓ Correlation heatmap
5. ✓ Risk score distribution
6. ✓ Seasonal comparisons
7. ✓ Risk trends over time
8. ✓ Stress events analysis

## Next Steps

Proceed to **Notebook 05: Scenario Modeling** to:
- Simulate +2°C warming scenario
- Project future risks
- Create climate impact assessments